# Unit 2 Assignment: Building a Mixture of Experts (MoE) Router

**Topic:** Advanced Architecture using Groq API   
**Tools:** Python, Groq API, Dotenv

---

## 🎯 Objective

Build a **"Smart Customer Support Router"** using a **Mixture of Experts (MoE)** architecture.

Instead of one generalist AI handling everything, we create:
1. A **Router** — classifies the user's intent.
2. **Expert Agents** — each with a specialized system prompt.
3. An **Orchestrator** — ties it all together.

```
User Query
    │
    ▼
[ Router LLM ]  ──►  "technical" / "billing" / "general"
    │
    ▼
[ Expert System Prompt ] + [ User Query ]
    │
    ▼
[ Expert LLM Response ]
```

---

## Step 1: Install Dependencies

In [ ]:
%pip install groq python-dotenv --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 2.9 MB/s eta 0:00:00


---

## Step 2: Setup & API Key

Just like in the LangChain notebooks, we **never hardcode API keys**.  
We use `python-dotenv` to load from a `.env` file, or `getpass` as a fallback.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
import getpass
from groq import Groq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Base model — we simulate 'experts' via different system prompts, not different models
BASE_MODEL = "llama-3.1-8b-instant"

print("✅ Groq client initialized.")

✅ Groq client initialized.


---

## Step 3: Define the Expert Configurations

Recall from **Prompt Engineering (Notebook 2)** that a system prompt acts like a **role + constraints** block.  
Here, each "expert" is the *same model* but with a completely different system prompt — this is the core trick of MoE via prompting.

> 💡 **Key Insight:** Just like `temperature=0` gave us consistency in the LangChain notebooks,  
> we use `temperature=0` for the **Router** (deterministic classification) and `temperature=0.7` for **Experts** (natural responses).

In [ ]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Senior Technical Support Engineer.
Your role is to diagnose and fix software bugs, errors, and technical issues with precision.
- Always identify the root cause before suggesting a fix.
- Provide code snippets where applicable.
- Use technical terminology accurately.
- Be concise and structured (use numbered steps if needed).
- Do NOT provide billing or sales advice.""",
        "temperature": 0.7,
    },
    "billing": {
        "system_prompt": """You are a Billing Support Specialist.
Your role is to help customers with payment issues, refunds, subscription changes, and invoices.
- Always be empathetic and patient — billing issues cause real stress.
- Clearly explain refund policies and timelines.
- Do NOT attempt to answer technical bugs or sales questions.
- Reassure the customer that their issue will be resolved.""",
        "temperature": 0.7,
    },
    "general": {
        "system_prompt": """You are a friendly and helpful general customer support agent.
You handle casual questions, greetings, and queries that don't fit technical or billing categories.
- Keep responses warm, conversational, and brief.
- If the user seems to have a technical or billing issue, gently suggest the right team.""",
        "temperature": 0.8,
    },
}

print("✅ Expert configurations loaded.")
print(f"   Available experts: {list(MODEL_CONFIG.keys())}")

✅ Expert configurations loaded.
   Available experts: ['technical', 'billing', 'general']


---

## Step 4: The Router — `route_prompt()`

This is the **core of MoE**. The router is an LLM call that acts as a **classifier**.  

Notice how this mirrors the **Zero-Shot prompting** concept from Notebook 2 — we give the model a clear instruction and constrain its output to a single word.

In [ ]:
def route_prompt(user_input: str) -> str:
    """
    Classifies the user query into one of the expert categories.

    Args:
        user_input: The raw query from the user.

    Returns:
        A string: 'technical', 'billing', or 'general'.
        Defaults to 'general' if classification is ambiguous.
    """
    routing_system_prompt = """You are an intent classification engine.
Your ONLY job is to classify the user's message into exactly one of these categories:
  - technical
  - billing
  - general

Rules:
- 'technical': bug reports, errors, crashes, code issues, API problems.
- 'billing': payments, charges, refunds, subscriptions, invoices.
- 'general': greetings, general questions, anything else.

Respond with ONLY the single category word. No explanation. No punctuation."""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": routing_system_prompt},
            {"role": "user", "content": user_input},
        ],
        temperature=0,  # Deterministic — we want consistent classification
        max_tokens=10,  # The answer is just one word
    )

    category = response.choices[0].message.content.strip().lower()

    # Validate and fall back to 'general' if the model returns something unexpected
    if category not in MODEL_CONFIG:
        print(f"   ⚠️  Unexpected category '{category}' — defaulting to 'general'.")
        category = "general"

    return category


print("✅ Router function defined.")

✅ Router function defined.


### Quick Test: Does the Router Classify Correctly?

In [ ]:
test_queries = [
    "My Python script is throwing an IndexError on line 5.",
    "I was charged twice for my subscription this month.",
    "Hey, how are you doing today?",
]

print("--- Router Classification Test ---\n")
for q in test_queries:
    category = route_prompt(q)
    print(f"Query   : {q}")
    print(f"Category: {category}")
    print()

--- Router Classification Test ---

Query   : My Python script is throwing an IndexError on line 5.
Category: technical

Query   : I was charged twice for my subscription this month.
Category: billing

Query   : Hey, how are you doing today?
Category: general



---

## Step 5: The Orchestrator — `process_request()`

The orchestrator is the **main controller**. It:
1. Calls the router to get a category.
2. Looks up the right expert config.
3. Calls the LLM with the expert's system prompt.
4. Returns the final answer.

Think of it as the **pipeline** pattern from Notebook 1 — `Input → Prompt → LLM → Output`.

In [ ]:
def process_request(user_input: str) -> str:
    """
    Orchestrates the full MoE pipeline.

    Args:
        user_input: The raw query from the user.

    Returns:
        The expert's response as a string.
    """
    print(f"📥 User Input : {user_input}")

    # Step 1: Route to the right expert
    category = route_prompt(user_input)
    print(f"🔀 Routed to  : [{category.upper()} EXPERT]")

    # Step 2: Load the expert configuration
    expert_config = MODEL_CONFIG[category]

    # Step 3: Call the LLM with the expert's system prompt
    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": expert_config["system_prompt"]},
            {"role": "user", "content": user_input},
        ],
        temperature=expert_config["temperature"],
        max_tokens=512,
    )

    answer = response.choices[0].message.content.strip()
    return answer


print("✅ Orchestrator function defined.")

✅ Orchestrator function defined.


---

## Step 6: Run the Full MoE System

Let's test the complete pipeline end-to-end.

In [ ]:
# --- Test 1: Technical Query ---
print("=" * 60)
result = process_request("My Python script is throwing an IndexError on line 5.")
print(f"\n💬 Expert Response:\n{result}")
print("=" * 60)

📥 User Input : My Python script is throwing an IndexError on line 5.
🔀 Routed to  : [TECHNICAL EXPERT]

💬 Expert Response:
I'd be happy to help you troubleshoot the issue. 

To start, could you please provide more details about the error message, such as:

1. The full error message, including any stack traces or error codes.
2. The relevant code snippet from lines 1-10, especially around line 5.
3. The input data or environment that's causing the error.

This information will help me identify the root cause of the issue and provide a more accurate solution.

In general, the `IndexError` occurs when you're trying to access an element at an index that doesn't exist in the list or sequence.

Here's a general example of an `IndexError`:

```python
# Example code with IndexError
my_list = [1, 2, 3]
print(my_list[3])  # Raises IndexError: list index out of range
```

Let's get started with your specific issue.


In [ ]:
# --- Test 2: Billing Query ---
print("=" * 60)
result = process_request("I was charged twice for my subscription this month.")
print(f"\n💬 Expert Response:\n{result}")
print("=" * 60)

📥 User Input : I was charged twice for my subscription this month.
🔀 Routed to  : [BILLING EXPERT]

💬 Expert Response:
I'm so sorry to hear that you've been charged twice for your subscription. That can be really frustrating and stressful.

Firstly, let me reassure you that I'm here to help and we'll get this issue sorted out as soon as possible. Can you please provide me with your account information, including your email address or username associated with your subscription? This will help me locate your account and look into the duplicate charge.

Once I have that information, I'll check on our end to see what happened and work on reversing the duplicate charge as soon as possible. You can expect a full refund for the second charge within 3-5 business days, depending on your payment method.

In the meantime, if you'd like to avoid any further charges for the current month, I can also assist you with updating your subscription to cancel or pause it temporarily until we resolve the is

In [ ]:
# --- Test 3: General Query ---
print("=" * 60)
result = process_request("Hey! What services do you guys offer?")
print(f"\n💬 Expert Response:\n{result}")
print("=" * 60)

📥 User Input : Hey! What services do you guys offer?
🔀 Routed to  : [GENERAL EXPERT]

💬 Expert Response:
We're glad you're interested in our services. We offer a wide range of products, from home internet to TV streaming, phone plans, and even security systems for your home. If you're looking for something specific, feel free to let me know and I'll be happy to help you explore our options. What sounds interesting to you?


---

## 🚀 Bonus Challenge: Tool-Use Expert (Mock Fetch)

Can you handle queries that need **real-time data** (like crypto prices)?  
Instead of a hallucinated answer, we route to a function that fetches (or mocks) data.

This mirrors real **Tool Calling / Function Calling** patterns.

In [ ]:
import random


# --- Mock Tool ---
def mock_fetch_crypto_price(coin: str) -> str:
    """
    Simulates a real-time crypto price fetch.
    In production, replace this with a live API call (e.g., CoinGecko).
    """
    prices = {
        "bitcoin": round(random.uniform(60000, 70000), 2),
        "ethereum": round(random.uniform(3000, 4000), 2),
        "solana": round(random.uniform(140, 200), 2),
    }
    coin_lower = coin.lower()
    if coin_lower in prices:
        return f"The current mock price of {coin.capitalize()} is ${prices[coin_lower]:,.2f} USD."
    return f"Sorry, I don't have price data for '{coin}' right now."


# --- Updated Router with 'crypto' category ---
def route_prompt_v2(user_input: str) -> str:
    """Extended router that also detects crypto/price queries."""
    routing_system_prompt = """You are an intent classification engine.
Classify the user's message into exactly one of these categories:
  - technical
  - billing
  - crypto
  - general

Rules:
- 'technical': bug reports, errors, crashes, code issues.
- 'billing': payments, charges, refunds, subscriptions.
- 'crypto': questions about cryptocurrency prices (Bitcoin, Ethereum, etc.).
- 'general': anything else.

Respond with ONLY the single category word. No punctuation."""

    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": routing_system_prompt},
            {"role": "user", "content": user_input},
        ],
        temperature=0,
        max_tokens=10,
    )
    category = response.choices[0].message.content.strip().lower()
    valid = {"technical", "billing", "crypto", "general"}
    return category if category in valid else "general"


# --- Updated Orchestrator with Tool-Use Branch ---
def process_request_v2(user_input: str) -> str:
    """Extended orchestrator that handles the 'crypto' tool-use expert."""
    print(f"📥 User Input : {user_input}")

    category = route_prompt_v2(user_input)
    print(f"🔀 Routed to  : [{category.upper()} EXPERT]")

    # Tool-Use branch: bypass LLM, call a function directly
    if category == "crypto":
        # Extract coin name with a quick LLM call
        coin_extraction = client.chat.completions.create(
            model=BASE_MODEL,
            messages=[
                {"role": "system", "content": "Extract the cryptocurrency name from the text. Return ONLY the coin name in lowercase (e.g., bitcoin, ethereum, solana). No other text."},
                {"role": "user", "content": user_input},
            ],
            temperature=0,
            max_tokens=10,
        )
        coin = coin_extraction.choices[0].message.content.strip().lower()
        print(f"🔧 Tool call  : mock_fetch_crypto_price('{coin}')")
        return mock_fetch_crypto_price(coin)

    # Standard LLM expert branch
    expert_config = MODEL_CONFIG.get(category, MODEL_CONFIG["general"])
    response = client.chat.completions.create(
        model=BASE_MODEL,
        messages=[
            {"role": "system", "content": expert_config["system_prompt"]},
            {"role": "user", "content": user_input},
        ],
        temperature=expert_config["temperature"],
        max_tokens=512,
    )
    return response.choices[0].message.content.strip()


print("✅ Bonus (v2) functions defined.")

✅ Bonus (v2) functions defined.


In [ ]:
# --- Bonus Test: Crypto Tool-Use ---
print("=" * 60)
result = process_request_v2("What is the current price of Bitcoin?")
print(f"\n💬 Expert Response:\n{result}")
print("=" * 60)

print()

# --- Bonus Test: Still works for normal queries ---
print("=" * 60)
result = process_request_v2("I can't log into my account, it says invalid credentials.")
print(f"\n💬 Expert Response:\n{result}")
print("=" * 60)

📥 User Input : What is the current price of Bitcoin?
🔀 Routed to  : [CRYPTO EXPERT]
🔧 Tool call  : mock_fetch_crypto_price('i can't provide real-time information. however,')

💬 Expert Response:
Sorry, I don't have price data for 'i can't provide real-time information. however,' right now.

📥 User Input : I can't log into my account, it says invalid credentials.
🔀 Routed to  : [TECHNICAL EXPERT]

💬 Expert Response:
Invalid credentials can be caused by several factors. To troubleshoot this issue, let's go through a step-by-step process:

1. **Clear Browser Cache and Cookies**: Sometimes, cached data can cause issues. Clear your browser's cache and cookies, then try logging in again.
   - For Chrome: Press `Ctrl + Shift + R` (Windows/Linux) or `Cmd + Shift + R` (Mac) to reload the page, and then try logging in.
   - For Firefox: Press `Ctrl + Shift + R` (Windows/Linux) or `Cmd + Shift + R` (Mac) to reload the page, and then try logging in.

2. **Check Autocomplete Settings**: Autocomplete

---

## ✅ Summary

| Component | Function | Key Parameter |
|---|---|---|
| **Router** | `route_prompt()` | `temperature=0` (consistent) |
| **Expert (Technical)** | System prompt in `MODEL_CONFIG` | `temperature=0.7` |
| **Expert (Billing)** | System prompt in `MODEL_CONFIG` | `temperature=0.7` |
| **Expert (General)** | System prompt in `MODEL_CONFIG` | `temperature=0.8` |
| **Orchestrator** | `process_request()` | Ties everything together |
| **Bonus Tool Expert** | `mock_fetch_crypto_price()` | Bypasses LLM entirely |

**Concepts used from earlier notebooks:**
- `temperature=0` for deterministic classification (Notebook 1a)
- System prompts as role + constraint blocks (Notebook 1b)
- CO-STAR style structured prompts (Notebook 2a)
- Zero-shot classification (Notebook 2b)